# XGBoost risk model: template

Trains an XGBoost default-risk model on the same data and the same train/validation/test split the app uses,
then exports it so the app can load it. **Do not touch the test split in this notebook**: the app evaluates on it
once, after calibration, so the reported numbers stay honest.

Contract (details in `docs/MODEL_CONTRACT.md`):
* train on `sp.X_tr` only (validation may be used for early stopping)
* export `model.json` + `model_card.json` into `models/incoming/`
* then run `python -m drishti.ml.import_model --load` from the repo root

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, "..")          # run this notebook from notebooks/; repo root must be importable

from drishti.ml.data import FEATURES, load_raw
from drishti.ml.train import make_splits

raw = load_raw()                  # data/raw/ (run `make data` once to download it)
sp = make_splits(raw)             # same 60/20/20 stratified split, seed 42
print(sp.X_tr.shape, sp.X_val.shape, "default rate (train):", sp.y_tr.mean().round(3))

## Features

The app computes the engineered features in `drishti/ml/data.py::add_features`. To use a new feature, add it there
(so the API and the agents compute it too), then list it in `sp.features` below.

In [ ]:
sp = make_splits(raw, FEATURES)   # replace FEATURES with your own list if you add features
sp.features

## Train

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=2000, learning_rate=0.03, max_depth=4, subsample=0.9, colsample_bytree=0.9,
    reg_lambda=1.0, eval_metric="aucpr", early_stopping_rounds=50, tree_method="hist", random_state=42,
)
model.fit(sp.X_tr, sp.y_tr, eval_set=[(sp.X_val, sp.y_val)], verbose=False)
print("best iteration:", model.best_iteration)

## Check on validation only

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

p_val = model.predict_proba(sp.X_val)[:, 1]
print("validation PR-AUC:", round(average_precision_score(sp.y_val, p_val), 4),
      "| ROC-AUC:", round(roc_auc_score(sp.y_val, p_val), 4))

## Export for the app

In [ ]:
out = pathlib.Path("../models/incoming")
out.mkdir(parents=True, exist_ok=True)
model.save_model(out / "model.json")
(out / "model_card.json").write_text(json.dumps({
    "name": "xgboost-notebook-v1",
    "features": sp.features,
    "trained_on": "app train split only (seed 42); validation used for early stopping",
    "author": "",
    "notes": "",
}, indent=2))
print("exported to", out.resolve())

## Next

From the repo root:

```bash
python -m drishti.ml.import_model --load
```
It validates the model, calibrates it, evaluates it once on the held-out test split, writes `models/metrics.json`
and re-scores every customer. Commit `models/incoming/model.json` and `model_card.json` so everyone (and the Docker build) uses it.